# VisWord 03 — Text-only + multimodal baselines (rows 16–20)

| # | Model | Input | Purpose |
|---|---|---|---|
| 16 | `bert-base-uncased` | page title | Text baseline |
| 17 | BERT | title + body[:200] | Text + context |
| 18 | `all-MiniLM-L6-v2` | title | Fast text baseline |
| 19 | CLIP text branch | title | Text in CLIP space |
| 20 | CLIP cross-modal | image + title | Zero-shot multimodal |

Text-only rows (16–19) run Phase-2 anchor retrieval only (text-only is
meaningless for a Phase-1 image-crop query). Row 20 runs both phases.

The anchor dataset's `metadata.jsonl` provides `visible_text_snippet` and `title` per image, which we use as the text per pool element.

**Runtime:** T4 is enough.  **Wallclock:** ~20 min total.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json
from pathlib import Path
from datetime import datetime
PROJECT = '/content/drive/MyDrive/VISWORD'
REPO_DIR = '/content/VISWORD'
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hkanpak21/VISWORD.git $REPO_DIR
sys.path.insert(0, f'{REPO_DIR}/src')
%cd $REPO_DIR
import torch, torch.nn.functional as F
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load anchor metadata and triplets

In [ ]:
anchors_root = Path(PROJECT) / 'data' / 'wiki_ss_anchors'
meta = {}
for line in open(anchors_root / 'metadata.jsonl'):
    r = json.loads(line)
    meta[r['image_path']] = r
print('anchor metadata entries:', len(meta))

triplets = [json.loads(l) for l in open(anchors_root / 'triplets_val.jsonl').read().splitlines() if l.strip()]
print('val triplets:', len(triplets))

def text_for(image_fname, mode='title'):
    m = meta.get(image_fname)
    if m is None: return ''
    title = m.get('title', '').replace('_', ' ')
    if mode == 'title': return title
    if mode == 'title_body': return (title + '\n' + (m.get('visible_text_snippet', '') or ''))[:2000]
    return title

## Shared Phase-2 text-retrieval helper

In [ ]:
import numpy as np

def phase2_text_retrieval(encode_texts, row_label, mode='title', k_values=(1,5,10,20), max_triplets=200):
    run_dir = Path(PROJECT) / 'runs' / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_{row_label}'
    run_dir.mkdir(parents=True, exist_ok=True)
    scores = {k: 0 for k in k_values}
    same_sim, diff_sim, n_valid = [], [], 0
    for t in triplets[:max_triplets]:
        pos, neg = t['positives'], t['negatives']
        anc = t['anchor']
        if not pos or not neg: continue
        # skip if anchor image is actually missing (upstream HF only ships ~10k)
        if not (anchors_root / 'images' / anc).exists(): continue
        a_text = text_for(anc, mode)
        pool_texts = [text_for(p, mode) for p in pos + neg]
        if not a_text or not any(pool_texts): continue
        n_valid += 1
        a_emb = encode_texts([a_text])
        p_emb = encode_texts(pool_texts)
        sim = (a_emb @ p_emb.T).squeeze(0)
        for k in k_values:
            topk = sim.topk(min(k, len(sim))).indices
            if any(i < len(pos) for i in topk.tolist()): scores[k] += 1
        same_sim.extend(sim[:len(pos)].tolist())
        diff_sim.extend(sim[len(pos):].tolist())
    for k in k_values: scores[k] /= max(n_valid, 1)
    out = {
        'checkpoint': None, 'checkpoint_step': None,
        'num_triplets': n_valid,
        'text_mode': mode,
        'recall': {str(k): scores[k] for k in k_values},
        'sanity': {
            'same_sim_mean': float(np.mean(same_sim)) if same_sim else 0,
            'diff_sim_mean': float(np.mean(diff_sim)) if diff_sim else 0,
            'gap': float(np.mean(same_sim)-np.mean(diff_sim)) if same_sim and diff_sim else 0,
        }
    }
    json.dump(out, open(run_dir / 'phase2_recall.json', 'w'), indent=2)
    print(f'{row_label}: P2 R@1={scores[1]:.3f}  R@5={scores[5]:.3f}  gap={out["sanity"]["gap"]:+.3f}  (n={n_valid})')
    return run_dir, out

## Row 18 — sentence-MiniLM (fastest, run first)

In [ ]:
from sentence_transformers import SentenceTransformer
minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2').to(device)
def minilm_encode(texts):
    with torch.no_grad():
        e = minilm.encode(texts, convert_to_tensor=True, device=device)
    return F.normalize(e, dim=-1).cpu()
phase2_text_retrieval(minilm_encode, 'row18_minilm_title', mode='title')

## Row 16 — BERT-base mean-pooled, title only

In [ ]:
from transformers import AutoTokenizer, AutoModel
tok = AutoTokenizer.from_pretrained('bert-base-uncased')
bert = AutoModel.from_pretrained('bert-base-uncased').to(device).eval()
def bert_encode(texts, max_length=64):
    enc = tok(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(device)
    with torch.no_grad():
        out = bert(**enc).last_hidden_state  # (B, T, 768)
        mask = enc['attention_mask'].unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1).clamp_min(1)
    return F.normalize(pooled, dim=-1).cpu()
phase2_text_retrieval(bert_encode, 'row16_bert_title', mode='title')

## Row 17 — BERT with title + body[:200]

In [ ]:
def bert_encode_long(texts): return bert_encode(texts, max_length=256)
phase2_text_retrieval(bert_encode_long, 'row17_bert_title_body', mode='title_body')

## Row 19 — CLIP text branch

In [ ]:
import open_clip
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
clip_model = clip_model.to(device).eval()
clip_tok = open_clip.get_tokenizer('ViT-B-16')
def clip_text_encode(texts):
    tokens = clip_tok(texts).to(device)
    with torch.no_grad():
        e = clip_model.encode_text(tokens).float()
    return F.normalize(e, dim=-1).cpu()
phase2_text_retrieval(clip_text_encode, 'row19_clip_text', mode='title')

## Row 20 — CLIP cross-modal (image query → text pool and vice versa)

Special protocol: anchor is an **image**, pool items are **text descriptions**. This is CLIP's designed use-case. Separately we also run image→image (matching row 6).

In [ ]:
from PIL import Image
def clip_image_encode(image_paths):
    imgs = [clip_preprocess(Image.open(p).convert('RGB')) for p in image_paths]
    x = torch.stack(imgs).to(device)
    with torch.no_grad():
        return F.normalize(clip_model.encode_image(x).float(), dim=-1).cpu()

run_dir = Path(PROJECT) / 'runs' / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_row20_clip_crossmodal'
run_dir.mkdir(parents=True, exist_ok=True)
scores = {k: 0 for k in (1, 5, 10, 20)}; n_valid = 0
for t in triplets[:200]:
    pos, neg, anc = t['positives'], t['negatives'], t['anchor']
    if not pos or not neg or not (anchors_root / 'images' / anc).exists(): continue
    valid_pool = [p for p in pos + neg if (anchors_root / 'images' / p).exists()]
    if len(valid_pool) < 2: continue
    n_valid += 1
    a_img = clip_image_encode([anchors_root / 'images' / anc])
    pool_texts = [text_for(p, 'title') for p in valid_pool]
    p_txt = clip_text_encode(pool_texts)
    sim = (a_img @ p_txt.T).squeeze(0)
    n_pos_in_pool = sum(1 for p in pos if p in valid_pool)
    for k in scores:
        topk = sim.topk(min(k, len(sim))).indices
        if any(i < n_pos_in_pool for i in topk.tolist()): scores[k] += 1
for k in scores: scores[k] /= max(n_valid, 1)
json.dump({'num_triplets': n_valid, 'recall': {str(k): v for k, v in scores.items()}, 'protocol': 'image_query_text_pool'},
          open(run_dir / 'phase2_recall.json', 'w'), indent=2)
print(f'row20_clip_crossmodal: P2 R@1={scores[1]:.3f}  R@10={scores[10]:.3f}  (n={n_valid})')

## Summary table (rows 16-20)

In [ ]:
import pandas as pd
rows_out = []
for d in sorted(Path(f'{PROJECT}/runs').glob('*_row1[6-9]_*')) + list(Path(f'{PROJECT}/runs').glob('*_row20_*')):
    r = json.load(open(d / 'phase2_recall.json'))
    rows_out.append({
        'row': d.name.split('_row')[1][:2],
        'label': d.name.split('_', 2)[-1],
        'P2_R@1': r['recall']['1'],
        'P2_R@10': r['recall']['10'],
        'n': r['num_triplets'],
    })
df = pd.DataFrame(rows_out)
print(df.to_string(index=False))
df.to_csv(f'{PROJECT}/runs/zeroshot_text_summary.csv', index=False)